# SICKLE — exploratory data analysis

Read-only. Nothing here trains anything or writes into `outputs/`.

The three facts this notebook exists to make concrete, because each one
constrains what the study can claim:

1. **A plot occupies a tiny fraction of its chip** — only ~3% of pixels carry a label.
2. **Cloud cover removes a large share of mid-season optical frames.**
3. **The test split is unlabelled**, so every number is a validation number.


In [ ]:
import sys, json, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
PROJ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJ))
from src import utils
cfg = utils.load_config('v2')
ROOT = Path(cfg['data']['root'])
print('data root:', ROOT)
print('exists   :', ROOT.exists())


## 1. The tabular index


In [ ]:
df = pd.read_csv(ROOT / cfg['data']['tabular'])
print(df.shape)
print(df.SPLIT.value_counts(dropna=False).to_string())
df.head()


### The test split is unlabelled

Not a convention — the columns are literally empty, and no masks exist on disk.


In [ ]:
for sp in ['train','val','test']:
    s = df[df.SPLIT == sp]
    print(f'{sp:6s} rows {len(s):5d}  PLOT_ID non-null {s.PLOT_ID.notna().sum():5d}'
          f'  CROP non-null {s.CROP.notna().sum() if "CROP" in s else -1:5d}')


## 2. Plots, not samples, are the unit

One plot contributes several season-level samples, so sample counts overstate
the independent evidence by roughly a factor of seven.


In [ ]:
print('unique plots :', df.PLOT_ID.nunique())
print('samples      :', len(df))
print('samples/plot :', round(len(df)/max(df.PLOT_ID.nunique(),1), 2))
for sp in ['train','val']:
    s = df[df.SPLIT == sp]
    print(f'{sp:6s} {len(s):5d} samples / {s.PLOT_ID.nunique():4d} plots')


## 3. How much of a chip is actually labelled?

This is the single most limiting property of the dataset.


In [ ]:
sys.path.insert(0, str(PROJ / 'shims')); sys.path.insert(0, str(PROJ / 'reference_impl'))
from src.dataset import build_loader
loader, vdf = build_loader(cfg, 'val', 'eval')
ig = cfg['data']['ignore_index']
fracs = []
for i, (inp, mask) in enumerate(loader):
    y = mask['crop_type'].numpy()
    fracs += [(yi != ig).mean() for yi in y]
    if i >= 2: break
fracs = np.array(fracs)
print(f'labelled pixel fraction: mean {fracs.mean():.3f}  median {np.median(fracs):.3f}'
      f'  min {fracs.min():.3f}  max {fracs.max():.3f}')
plt.figure(figsize=(6,3)); plt.hist(fracs, bins=30)
plt.xlabel('fraction of chip carrying a label'); plt.ylabel('samples'); plt.tight_layout()


## 4. Class balance at plot level

Paddy is the plot-level majority, yet it is the **harder** class in every one of
the twelve released cells, often by 20–30 IoU points. Macro IoU alone hides that.


In [ ]:
if 'PADDY_BIN' in df.columns:
    print(df.groupby('SPLIT').PADDY_BIN.value_counts().to_string())


## 5. Cloud cover

A frame stored as all-zero is a missing/clouded acquisition. These are also
exactly the frames that trigger the padding defects in `docs/MODEL_DEFECTS.md`.


In [ ]:
empty = tot = 0
for i, (inp, mask) in enumerate(loader):
    img = inp[list(cfg['satellites'])[0]][0].numpy()
    for s in img:
        for t in s:
            tot += 1
            if not np.any(t): empty += 1
    if i >= 1: break
print(f'all-zero frames: {empty}/{tot} = {100*empty/max(tot,1):.1f}%')


---

### What this means for the study

* ~3% labelled pixels per chip → **pixel-pooled IoU is dominated by the few
  largest plots**, which is why Plot-Macro IoU is reported beside it.
* 36 validation plots → **bootstrap CIs resample plots, never pixels**.
* Unlabelled test split → **validation numbers only**, not comparable to the
  paper's published test table.
